# **GROUP D - HKBU Study Companion**
## Project Members：
**XU Xin 25409816** <br>
**CHEN Yapeng 25420143**<br>
**Zhang Yuming 25401939** 



## **Readme:**
#### 1.Use the following statement to install the corresponding dependent libraries：
! pip install torch numpy pandas PyPDF2 docx sentence-transformers scikit-learn ollama tiktoken
#### 2.Use the following statements to install two ollama models, "gemma3:12b" and "minimax-m2.5:cloud". 
#### The former is the local deployment model and the latter is the judge result model,and remember to turn on ollama serve.
ollama pull gemma3:12b <br>
ollama pull minimax-m2.5:cloud
#### 3.Place the data files in the `data` folder located in the same directory as the `.ipynb` file.

## **Dependency Library**
`os`:Mainly used for traversing directories and joining file paths to batch read local documents.<br>
`numpy`,`pandas`,`PyPDF2`,`json`,`re `,`docx`:Extracts raw data from 'PDF','docx','excel' documents to serve as the knowledge base source.<br>
<br>
`sklearn`:<br>
`sklearn.feature_extraction.text.TfidfVectorizer`:Used for  lexical retrieval. It measures word importance using Term Frequency and Inverse Document Frequency, suitable for exact keyword matching.<br>
`sklearn.metrics.pairwise.cosine_similarity`: Used to calculate the cosine similarity between two vectors. In semantic retrieval, it measures how similar the "query vector" is to the "document chunk vector".<br>
`sklearn.preprocessing.MinMaxScaler`: Used for feature scaling. In hybrid retrieval, scores from keyword retrieval (e.g., TF-IDF scores) and semantic retrieval (similarity between 0-1) have different scales. This tool normalizes them to the same range so they can be combined.<br>
`sentence_transformers`:Used for neural retrieval. It transforms your documents and user queries into vectors so the computer can understand semantic similarity.<br>
<br>
`ollama`:Used for running and calling Large Language Models locally.<br>
`time`:Used to calculate execution time.<br>
`tiktoken` :Used to precisely count tokens.<br>

In [1]:
import os
import torch 
import numpy as np
import pandas as pd
import PyPDF2
import json 
import re  
from docx import Document

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

import ollama
import time
import tiktoken

## **Model Configuration**
### Include the `Used Model` , `File Path`, `Model Chunking Parameters`, `Retrieved Files Number`,  `Model's Temperature` and `Context Window`.

In [2]:
# RAG Configuration
LOCAL_MODEL = "gemma3:12b" 
OLLAMA_HOST = "http://localhost:11434" 

# File path
DATA_DIR = "./data"
CHROMA_DB_DIR = "./chroma_db" 

# RAG Configuration
CHUNK_SIZE = 300 
CHUNK_OVERLAP = 50 
TOP_K_CONTEXT = 4 

temperature=0.3
num_ctx=4096

## **1. Data Loading and Chunking Module**
### Mainly responsible for data loading and data chunking.

### **Functions:**
#### `read_txt(file_path)`:Reads a `.txt` file.
#### `read_pdf(file_path)`:Reads a `.pdf` file.
#### `read_docx(file_path)`:Reads a `.docx` file.
#### `read_excel(file_path)`:Reads all sheets from `Excel` file,converts each row of data into an independent "JSON" object instead of simple text concatenation to ensures data integrity.

#### `load_and_chunk_data`:Automatically calls the corresponding read function based on the file extension (.txt, .pdf, .docx, .xlsx);Perform smart chunking on the data and return a list containing all processed text chunks.

In [3]:
# ==========================================
# 1. Data Loading and Chunking Module
# ==========================================

def read_txt(file_path):
    """Reads a TXT file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Failed to read TXT {file_path}: {e}")
        return ""

def read_pdf(file_path):
    """Reads a PDF file."""
    text = ""
    try:
        with open(file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Failed to read PDF {file_path}: {e}")
    return text

def read_docx(file_path):
    """Reads a DOCX file."""
    try:
        doc = Document(file_path)
        full_text = []
        for para in doc.paragraphs:
            full_text.append(para.text)
        return '\n'.join(full_text)
    except Exception as e:
        print(f"Failed to read DOCX {file_path}: {e}")
        return ""

def read_excel(file_path):
    """
    Reads an Excel file (.xlsx, .xls).
    Strategy: Reads all sheets and converts table content into structured JSON strings.
    """
    try:
        excel_data = pd.read_excel(file_path, sheet_name=None, dtype=str)
        full_json_list = []

        for sheet_name, df in excel_data.items():
            df.columns = df.columns.str.strip()
            # Returns each row as an independent JSON object instead of merging into one large JSON.
            # This facilitates slicing by "row/object" later.
            records = df.to_dict(orient='records')
            for record in records:
                # Convert each row to a JSON string
                json_str = json.dumps(record, ensure_ascii=False)
                full_json_list.append(json_str)

        return "\n".join(full_json_list) # Join with newlines, each line is a complete course object

    except Exception as e:
        print(f"Failed to read Excel {file_path}: {e}")
        return ""

def load_and_chunk_data(data_dir, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """
    Generic document loading and chunking function (Smart Fix Version).
    Core Logic: Prioritizes the integrity of JSON objects to avoid splitting a single course in half.
    """
    chunks = []

    # Traverse directory
    for file in os.listdir(data_dir):
        file_path = os.path.join(data_dir, file)

        if os.path.isdir(file_path):
            continue

        content = ""

        # Select reading method based on file extension
        if file.endswith(".txt"):
            content = read_txt(file_path)
        elif file.endswith(".pdf"):
            content = read_pdf(file_path)
        elif file.endswith(".docx"):
            content = read_docx(file_path)
        elif file.endswith(".xlsx") or file.endswith(".xls"):
            content = read_excel(file_path)
        else:
            continue

        if not content:
            continue

        # --- Smart Chunking Logic ---

        # 1. Attempt to split by JSON object (for Excel/JSON data)
        # Assumes each line or each {...} is an independent unit
        potential_objects = re.split(r'(?<=\})\s*(?=\{)|\n', content)

        current_chunk = ""

        for obj in potential_objects:
            obj = obj.strip()
            if not obj:
                continue

            # Skip if empty or just punctuation
            if obj in ['{', '}', ',']:
                continue

            # If current chunk is empty, fill it
            if len(current_chunk) == 0:
                current_chunk = obj
            # If adding this object doesn't exceed the limit, append it
            elif len(current_chunk) + len(obj) < chunk_size:
                current_chunk += "\n" + obj
            # If it exceeds the limit, save the current chunk and start a new one
            else:
                chunks.append(current_chunk)
                current_chunk = obj

        # Save the last chunk
        if len(current_chunk) > 0:
            chunks.append(current_chunk)

    print(f"Successfully loaded and chunked: {len(chunks)} document chunks.")
    return chunks

## **2. RAG Retriever Module**
### It takes a collection of document snippets and sets up 3 search systems:
#### (1)Lexical Search System: Matches based on the specific words used in your query.
#### (2)Neural Search System: Understands the true intent of your query, finding relevant content even if different words are used.
#### (3)Hybrid Search System:Combines the strengths of Lexical and Neural retrieval to achieve more comprehensive and accurate results.
### **Functions:**
#### `__init__`:This is the constructor, executed automatically when a RAGRetriever instance is created,prepare and load all retrieval tools.include"Data Preprocessing";"Initialize Keyword Retrieval";"Initialize Semantic Retrieval".
#### `retrieve_lexical`:Finds text snippets that are textually most similar to the query.
#### `retrieve_embedding`:Finds text snippets that are semantically closest in meaning to the query.
#### `retrieve_hybrid`:Combines the strengths of Lexical and Neural retrieval to achieve more comprehensive and accurate results,ensuring both exact keyword matching and broad semantic relevance, typically outperforming single retrieval methods

In [4]:
# ==========================================
# 2. Separate Retrieval Modes
# ==========================================
class RAGRetriever:
    def __init__(self, chunks):
        """
        Initializes the retriever, automatically compatible with List[str] or list of Document objects
        """
        self.chunks = chunks
        
        # --- 1. Data Preprocessing ---
        print(" Processing data format...")
        if not chunks:
            raise ValueError("Data chunks are empty!")
            
        # Check data type: use directly if string, otherwise extract .text
        if isinstance(chunks[0], str):
            self.texts = chunks
        else:
            self.texts = [chunk.text for chunk in chunks]

        # --- 2. Initialize TF-IDF (Keyword Retrieval) ---
        print(" Building TF-IDF matrix...")
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)

        # --- 3. Initialize Embedding (Semantic Retrieval) ---
        print(" Building Embedding vector store...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Note: Using a random vector here to simulate Embedding for code execution
        # In actual use, please replace with models like SentenceTransformer
        
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.model.encode(self.texts)
        

    def retrieve_lexical(self, query, top_k=TOP_K_CONTEXT):
        """Keyword Retrieval (TF-IDF)"""
        query_vec = self.vectorizer.transform([query])
        # Fix: cosine_similarity returns numpy array, no need for .toarray()
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        
        top_indices = scores.argsort()[-top_k:][::-1]
        return [(self.texts[i], float(scores[i])) for i in top_indices]

    def retrieve_embedding(self, query, top_k=TOP_K_CONTEXT):
        """Semantic Retrieval (Embedding)"""
        # Simulate query vectorization
        query_vec = self.model.encode([query], convert_to_numpy=True)
        
        scores = cosine_similarity(query_vec, self.embeddings).flatten()
        top_indices = scores.argsort()[-top_k:][::-1]
        return [(self.texts[i], float(scores[i])) for i in top_indices]

    def retrieve_hybrid(self, query, top_k=TOP_K_CONTEXT, alpha=0.5):
        """
        Hybrid Retrieval: Combines keywords and semantics
        alpha: 0.5 indicates 50% weight for each
        """
        # 1. Get candidate set (take Top 10 to ensure recall rate)
        lexical_results = self.retrieve_lexical(query, top_k=10)
        embedding_results = self.retrieve_embedding(query, top_k=10)

        # 2. Score Normalization and Fusion
        # Use dictionary to store final scores
        final_scores = {}
        
        # Extract all scores for normalization
        lex_scores = [score for _, score in lexical_results]
        emb_scores = [score for _, score in embedding_results]
        
        # Prevent division by zero, set default value if no results
        if not lex_scores: lex_scores = [0]
        if not emb_scores: emb_scores = [0]

        # Normalization (Min-Max Scaling)
        # Ensure both scores are between 0-1 with consistent magnitude
        lex_min, lex_max = min(lex_scores), max(lex_scores)
        emb_min, emb_max = min(emb_scores), max(emb_scores)

        def normalize(score, min_val, max_val):
            if max_val == min_val: return 0.5
            return (score - min_val) / (max_val - min_val)

        # Process keyword results
        for text, score in lexical_results:
            if text not in final_scores: final_scores[text] = 0
            norm_score = normalize(score, lex_min, lex_max)
            final_scores[text] += alpha * norm_score

        # Process semantic results
        for text, score in embedding_results:
            if text not in final_scores: final_scores[text] = 0
            norm_score = normalize(score, emb_min, emb_max)
            final_scores[text] += (1 - alpha) * norm_score

        # 3. Sort and return Top-K
        sorted_results = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
        return [(text, score) for text, score in sorted_results[:top_k]]

## **3. Memory Module**
### Manage and maintain the historical records of conversations to enable AI to "remember" previous chat contents, thereby achieving continuous multi-round conversations.
### **Functions:**
#### `__init__`:Initialize the memory module and set the capacity.
#### `add_message`:Save a new chat and automatically clear out expired old memories.
#### `get_formatted_history`:Convert the conversation records stored in the list into a readable text format.3

In [5]:
# ==========================================
# 3. Memory Module 
# ==========================================
class ConversationMemory:
    def __init__(self, max_history=3):
        """
        Initializes the memory module.
        :param max_history: Maximum number of conversation rounds to keep.
        """
        self.max_history = max_history
        self.history = []

    def add_message(self, role, content):
        """
        Adds a new message to the history and manages the sliding window.
        Automatically removes oldest messages if the limit is exceeded.
        """
        self.history.append({"role": role, "content": content})
        # Keep the last max_history * 2 messages (User + AI pairs)
        if len(self.history) > self.max_history * 2:
            self.history = self.history[-(self.max_history * 2):]

    def get_formatted_history(self):
        """
        Returns the conversation history as a formatted string.
        Format: "Role: Content"
        """
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.history])

In [6]:
# 初始化 Token 计算器 (cl100k_base 适配 Llama 3, Mistral, GPT-4 等)
try:
    token_encoder = tiktoken.get_encoding("cl100k_base")
except:
    token_encoder = tiktoken.get_encoding("p50k_base")

def count_tokens(text):
    return len(token_encoder.encode(text))

## **4. Generator Module**
### Responsible  for sending requests and for constructing complex prompts (include CoT) and managing generation parameters.
### **Functions:**
#### `__init__`:Initialize the generator,save the model name and default configurations.
#### `generate_response`:Sends the prompt to Ollama and receives the generated text
#### `build_rag_prompt`:Concatenates system instructions, reference context, conversation history, and the user's question into a complete Prompt.And we use "Few-Shot Prompting"and "CoT" improving accuracy and formatting consistency；reduces hallucinations and improves the handling of complex logic.
#### `count_tokens`:Used to count the Token of the text,help to understand the input limitations and billing of the model.

In [7]:
# ==========================================
# 4. Generator Module 
# ==========================================
try:
    token_encoder = tiktoken.get_encoding("cl100k_base")
except:
    token_encoder = tiktoken.get_encoding("p50k_base")
    
class OllamaGenerator:
    def __init__(self, LOCAL_MODEL, temperature, num_ctx): 
        self.model = LOCAL_MODEL
        # Save default configuration
        self.default_options = {"temperature": temperature, "num_ctx": num_ctx}

    def generate_response(self, final_prompt, temperature=None):
        """
        Generates a response from the model.
        :param final_prompt: The prompt string to send to the model.
        :param temperature: Optional, overrides default temperature (e.g., set to 0 for judge mode).
        """
        # Prepare configuration: override if temperature is provided, otherwise use default
        current_options = self.default_options.copy()
        if temperature is not None:
            current_options["temperature"] = temperature

        try:
            # Call Ollama API
            response = ollama.generate(
                model=self.model, 
                prompt=final_prompt, 
                options=current_options
            )
            return response['response']
        except Exception as e:
            return f"❌ Ollama Connection Error: {e}"

    def build_rag_prompt(self, query, context, chat_history=""):
        """
        Prompt Assembly: Assembles system instructions, history, and current query.
        """
        prompt = f"""
        # Role: HKBU Study Companion
        # Goal: Answer the question based on the provided Context. If no relevant information is found in the Context, answer "Cannot answer based on available materials".

        ## Example 1 (Direct Retrieval)
        # Follow the exact reasoning structure shown below. Keep the reasoning brief.

        # Context: 
        # [1] Course Policies: Lab assignments are due every Monday 13:30.
        # [2] Late submission without prior approval will not be accepted.

        # Question: What happens if I submit my lab assignment late?
        # Reasoning:
        # 1. Question asks about the consequence of late assignment submission.
        # 2. Context [2] states that late submissions are not accepted without prior approval.
        # 3. Formulate the answer in English and cite the source [2].
        # Answer: Late submissions are not accepted without prior approval [2].

        ## Example 2 (Conditional Logic & Synthesis)
        # Context: 
        # [3] From Year 2 of study onwards, all students are encouraged to schedule meetings with advisors as needed.
        # [4] In any year, students are required to meet with their advisor under the following circumstances: (i) when they declare or change a major; (ii) following an advisor change; (iii) when they are placed on academic probation; and (iv) when they opt out of TSM/SM.

        # Question: I am a Year 2 student. Is it mandatory for me to meet with my academic advisor?
        # Reasoning:
        # 1. Identify user status: Year 2 student.
        # 2. Check general rule: Context [3] says Year 2 students are only "encouraged" to meet, so it's not generally mandatory.
        # 3. Check exceptions: Context [4] states that for "any year", meetings are required under specific conditions like probation or changing majors.
        # 4. Synthesis: The answer must combine the general optional rule with the mandatory exceptions.
        # Answer: Generally, as a Year 2 student, it is not mandatory but highly encouraged to schedule meetings as needed [3]. However, you are required to meet with your advisor if you change your major, experience an advisor change, are placed on academic probation, or opt out of a TSM/SM [4].

        ## Context (Reference)
        {context}

        ## Conversation History (if any)
        {chat_history}

        ## Final Task
        Use the reasoning pattern shown above to answer the question.
        Question: {query}
        Answer:
        """
        return prompt

    # Tokenizer initialization (moved inside class for context, though usually defined globally)
    def count_tokens(self, text):
        """
        计算文本的 Token 数量
        """
        if not text:
            return 0
        return len(encoder.encode(text))

   

## **5. Judge Module**
### Judge the results and count the tokens.
### **Functions:**
#### `count_tokens`:Utilize another large models to evaluate the quality of the results generated by different retrieval strategies (lexical,neural,hybird)
#### `count_tokens`:Used to count the Token of the text,help to understand the input limitations and billing of the model.

In [8]:
# ==========================================
# 5. LLM as Judge 
# ==========================================
def judge(query,r_lex, r_emb, r_hyb):
    # We let the judge view all three responses simultaneously and select the best one
    judge_template = """
You are an expert evaluator for RAG (Retrieval-Augmented Generation) systems.

User Query: "{query}"

Please compare the following three responses generated based on different retrieval strategies:

[Response A - Lexical Retrieval]:
{resp_lex}

[Response B - Neural Retrieval]:
{resp_emb}

[Response C - Hybrid Retrieval]:
{resp_hyb}

Evaluate and score from 1 to 5 for each criterion based on the following strict definitions:
1) Fidelity (Factuality): Focus ONLY on semantic equivalence. Does the generated answer convey the same factual truth as the Gold Standard? Do NOT penalize for different wording, sentence structure, or the language used (Chinese vs. English).
2) Completeness: Does the generated answer cover all the critical information points present in the Gold Standard?
3) Restraint: Did the generated answer stay within the bounds of the question and the Gold Standard, or did it hallucinate extra, unverified information?

Please output only the winner (A, B, or C) along with a brief justification.
Format Example:
Winner: [Strategy Name]
Reason: ...
"""

    # Format the prompt with the actual data
    judge_prompt = judge_template.format(
        query=query,
        resp_lex=r_lex,
        resp_emb=r_emb,
        resp_hyb=r_hyb,
    )

    # Use cloud model as judge (temperature=0 ensures evaluation consistency)
    judge_resp = ollama.generate(
        model="minimax-m2.5:cloud",
        prompt=judge_prompt,
        options={"temperature": 0},
    )
    print(" ✅ Judge Model response received.") # 如果这行没打印，说明卡在上一行
    final_verdict = (judge_resp.get("response") or "").strip()

    print("\n \033[1m Final Verdict: \033[0m")
    print(final_verdict if final_verdict else "(Judge returned no content)")
    return 0


## **Main System Process**
### Phase 1: System Initialization

1.  **Data Preparation**: Load local documents, chunk them into smaller pieces, and build the index.
2.  **Component Startup**:
    -   **Retriever**: The "librarian" responsible for finding information.
    -   **Generator**: The "writer" responsible for drafting answers.
    -   **Memory**: The "secretary" responsible for taking notes.

###  Phase 2: Main Interaction Loop
The program enters a `while True` loop, waiting for your input.

#### 1. Receive Instruction & Read Memory
-   You input a question (e.g., "What is my homework for next week?").
-    **Key Action**: The system immediately checks `memory` for your last 5 rounds of conversation history. This is to allow the AI to understand context (e.g., if you previously asked "I am a Computer Science student").

#### 2. Three Strategies "Race" in Parallel
-   System launches three "avatars" to answer your question simultaneously for comparison:


-   **Contestant A (Lexical Retrieval)**: Like a traditional search engine, finding matches via word matching.
。
-   **Contestant B (Neural Retrieval)**: Finding matches by understanding sentence meaning (e.g., searching "how to fail" matches "consequences of failing").

-   **Contestant C (Hybrid Retrieval)**: Combines the strengths of A and B, usually yielding the best results.

#### 3. Execute Retrieval and Generation

1.  **Start Timer**.

2.  **Retrieve**: Search the database for the most relevant chunks.

3.  **Assemble Prompt**: Pack the **History** + **Found Chunks** + **Your Question** into a long text.

4.  **Generate**: Send to LLM to generate a response.

5.  **End Timer & Count Tokens**: Record time taken and Token usage.

6.  **Display**: Print the found chunks (with scores) and the generated answer on the screen.


#### 4. Data Aggregation & Statistics
The system prints a comparison table to compare the time and token usage of the three different methods

#### 5. Update Memory
-   The system selects the result from **Hybrid Retrieval** as the "standard answer".

-   It saves your question and this "standard answer" into `memory`. 


#### 6. LLM Judge Evaluation
Finally, the system activates "Judge Mode":

-   It sends all three answers (A, B, C) to the judge model.

-   The judge scores them based on criteria like accuracy and completeness, and selects the "Best Answer" for the round.




In [ ]:
# ==========================================
# 1. System Initialization
# ==========================================
print(" \033[1m Starting RAG System \033[0m")
    
# Load and chunk data 
chunks = load_and_chunk_data(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
# Retriever initialized
retriever = RAGRetriever(chunks)
# Generator initialized
generator = OllamaGenerator(LOCAL_MODEL, temperature, num_ctx)
# Memory initialized
memory = ConversationMemory()

print(f" System Ready | Model: {LOCAL_MODEL}")
print("="*130)

# ==========================================
# 2. Main Interaction Loop
# ==========================================
while True:
    try:
        user_input = input("\n  User  (Type 'exit' to quit): ")
        if user_input.lower() in ["exit", "quit"]:
            break
        if not user_input.strip():
            continue

        print(f"\n Retrieving for question: '{user_input}'...")

        # --- Get Historical Memory ---
        # Get formatted history (Class handles the limit internally)
        chat_history = memory.get_formatted_history() 

        results = {}

        # --- Define Three Retrieval Strategies ---
        strategies = [
            ("Lexical Retrieval", lambda q: retriever.retrieve_lexical(q, top_k=TOP_K_CONTEXT)),
            ("Neural Retrieval", lambda q: retriever.retrieve_embedding(q, top_k=TOP_K_CONTEXT)),
            ("Hybrid Retrieval", lambda q: retriever.retrieve_hybrid(q, top_k=TOP_K_CONTEXT, alpha=0.5)),
        ]

        # ==========================================
        # 3. Execute Retrieval, Display Chunks, Generate Response
        # ==========================================
        for strat_name, retrieve_fn in strategies:
            print(f"\n--- [{strat_name}] Retrieval Details ---")

            # Start Timer
            t0 = time.perf_counter()
            retrieved_items = retrieve_fn(user_input)

            if not retrieved_items:
                elapsed = time.perf_counter() - t0
                print(" No relevant chunks found.")
                results[strat_name] = {
                    "response": "No relevant content",
                    "context": "",
                    "tokens": 0,
                    "latency": elapsed,
                }
                print(f"    Total Time: {elapsed:.2f}s")
                continue

            # 1. Display chunk ranking and scores
            context_parts = []
            for i, (text, score) in enumerate(retrieved_items):
                short_text = text[:300].replace('\n', ' ') + "..."
                print(f"   #{i+1} [Score: {score:.4f}] {short_text}")
                context_parts.append(text)

            # 2. Build context and generate
            full_context = "\n\n".join(context_parts)
            
            prompt = generator.build_rag_prompt(user_input, full_context, chat_history)
            
            response = generator.generate_response(prompt)

            # End Timer
            elapsed = time.perf_counter() - t0

            # 3. Count Tokens
            total_tokens = count_tokens(prompt) + count_tokens(response)

            # 4. Save Results
            results[strat_name] = {
                "response": response,
                "context": full_context,
                "tokens": total_tokens,
                "latency": elapsed,
            }
            
            print(f"\n--  \033[1m Generated Response \033[0m --\n{response}\n... (Consumed {total_tokens} tokens) ...\n")
            print(f"Total Time: {elapsed:.2f}s")
            print("-" * 130)
        
        # ==========================================
        # 4. Final Statistics
        # ==========================================
        print("\n \033[1m Interaction Statistics: \033[0m")
        for name, data in results.items():
            print(f" - {name}: {data.get('tokens', 0)} tokens | {data.get('latency', 0.0):.2f}s")
        
        print("-" * 130)
        
        # ==========================================
        # 5. Update Memory (Critical Step)
        # ==========================================
        # Use Hybrid Retrieval result as the final answer for memory
        final_response = results.get("Hybrid Retrieval", {}).get("response", "No answer")
        
        # Add current Q&A pair to history
        memory.add_message("User", user_input)
        memory.add_message("Assistant", final_response)
        
        print(f" Conversation saved to memory (Current length: {len(memory.history)})")
        
        # ==========================================
        # 6. LLM Judge Evaluation
        # ==========================================
        print("\n" + "=" * 130)
        print(" \033[1m Starting LLM Judge for comparison... \033[0m")
        print("-" * 130)
        
        
        r_lex = results.get("Lexical Retrieval", {}).get("response", "No content")
        r_emb = results.get("Neural Retrieval", {}).get("response", "No content")
        r_hyb = results.get("Hybrid Retrieval", {}).get("response", "No content")
        
        judge(user_input,r_lex, r_emb, r_hyb)

    except KeyboardInterrupt:
        print("\n Program exited.")
        break
    except Exception as e:
        print(f"\n Error occurred: {e}")

  Starting RAG System 
Successfully loaded and chunked: 138 document chunks.
 Processing data format...
 Building TF-IDF matrix...
 Building Embedding vector store...
 System Ready | Model: gemma3:12b



  User  (Type 'exit' to quit):  I need advice of selecting course, who should I talk to?



 Retrieving for question: 'I need advice of selecting course, who should I talk to?'...

--- [Lexical Retrieval] Retrieval Details ---
   #1 [Score: 0.1918] To introduce staff to students. To build learning communities for students. To guide students to seek and use University learning resources and support. To advise students in selecting and planning for majors, if applicable, including first, second...
   #2 [Score: 0.1884] 3advisor in the home department2, who also gives advice on students’ first major. Students intending to opt for a TSM/SM should attend relevant briefings and seminars, with TSM/SM and major programme directors guiding Year 2 course selection. FYFD students of the individualised...
   #3 [Score: 0.1557] should be given to students so that they can update their study plans, relate what they have learnt to their endeavours after graduation, and formulate their career goals and/or plans for further studies. Advice can also be given to students who may encounter 